CELDA 1 — Instalación de librerías

In [ ]:
!pip install yfinance ta pymongo[srv] --quiet

CELDA 2 — Importaciones

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
from pymongo import MongoClient
from datetime import datetime, timezone
from google.colab import userdata

CELDA 3 — Configuración de tickers y conexión a Mongo

In [ ]:
# Los 5 activos definitivos del proyecto
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]

PERIODO = "2y"  # mismo periodo que usará el notebook de SVC

# Conexión a MongoDB Atlas usando el Secret configurado
MONGO_URI = userdata.get("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["ernesto_investing_ai"]
coleccion_precios = db["precios_ohlcv"]

# Verificar conexión
try:
    client.admin.command("ping")
    print("✅ Conexión a MongoDB Atlas exitosa.")
except Exception as e:
    print("❌ Error de conexión:", e)

✅ Conexión a MongoDB Atlas exitosa.


CELDA 4 — Función de descarga y corrección de MultiIndex

In [ ]:
def descargar_datos(ticker):
    """
    Descarga datos OHLCV de Yahoo Finance para un ticker.
    Corrige el MultiIndex que yfinance devuelve incluso para un solo ticker.
    """
    df = yf.download(ticker, period=PERIODO, auto_adjust=True, progress=False)

    if df.empty:
        raise ValueError(f"No se obtuvieron datos para {ticker}")

    # Corregir el MultiIndex de columnas
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    return df

CELDA 5 — Función de cálculo de indicadores técnicos

In [ ]:
def calcular_indicadores(df):
    """
    Calcula los indicadores técnicos definidos en el contrato de datos:
    SMA-20, SMA-50, EMA-12, EMA-26, RSI-14, MACD.
    """
    df = df.copy()

    df["SMA20"] = ta.trend.sma_indicator(df["Close"], window=20)
    df["SMA50"] = ta.trend.sma_indicator(df["Close"], window=50)
    df["EMA12"] = ta.trend.ema_indicator(df["Close"], window=12)
    df["EMA26"] = ta.trend.ema_indicator(df["Close"], window=26)
    df["RSI"] = ta.momentum.rsi(df["Close"], window=14)

    macd = ta.trend.MACD(df["Close"])
    df["MACD"] = macd.macd()

    # Eliminar filas iniciales sin indicadores (por ventanas móviles)
    df.dropna(inplace=True)

    return df

CELDA 6 — Función de transformación al contrato de datos (JSON/Mongo)

In [ ]:
def construir_documento(ticker, df):
    """
    Construye el documento con el esquema EXACTO del contrato de datos.
    Este mismo esquema lo usará la API y el frontend, sin transformaciones intermedias.
    """
    # Reemplazar NaN por None para que sea JSON/BSON válido
    df_limpio = df.replace({np.nan: None})

    documento = {
        "ticker": ticker,
        "fechas": df_limpio.index.strftime("%Y-%m-%d").tolist(),
        "open": df_limpio["Open"].round(4).tolist(),
        "high": df_limpio["High"].round(4).tolist(),
        "low": df_limpio["Low"].round(4).tolist(),
        "close": df_limpio["Close"].round(4).tolist(),
        "volume": df_limpio["Volume"].astype("Int64").tolist(),
        "sma20": [round(x, 4) if x is not None else None for x in df_limpio["SMA20"]],
        "sma50": [round(x, 4) if x is not None else None for x in df_limpio["SMA50"]],
        "ema12": [round(x, 4) if x is not None else None for x in df_limpio["EMA12"]],
        "ema26": [round(x, 4) if x is not None else None for x in df_limpio["EMA26"]],
        "rsi": [round(x, 4) if x is not None else None for x in df_limpio["RSI"]],
        "macd": [round(x, 4) if x is not None else None for x in df_limpio["MACD"]],
        "actualizado_en": datetime.now(timezone.utc)
    }

    return documento

CELDA 7 — Ejecución: descargar, procesar y guardar los 5 tickers

In [ ]:
resultados = {}

for ticker in TICKERS:
    print(f"Procesando {ticker}...")
    try:
        df_raw = descargar_datos(ticker)
        df_ind = calcular_indicadores(df_raw)
        documento = construir_documento(ticker, df_ind)

        # Upsert: si el ticker ya existe, lo actualiza; si no, lo crea
        coleccion_precios.update_one(
            {"ticker": ticker},
            {"$set": documento},
            upsert=True
        )

        resultados[ticker] = "✅ OK"
        print(f"  → {len(documento['fechas'])} registros guardados en MongoDB.")

    except Exception as e:
        resultados[ticker] = f"❌ ERROR: {e}"
        print(f"  → Error con {ticker}: {e}")

print("\nResumen final:")
for ticker, estado in resultados.items():
    print(f"  {ticker}: {estado}")

Procesando FSM...
  → 452 registros guardados en MongoDB.
Procesando VOLCABC1.LM...
  → 444 registros guardados en MongoDB.
Procesando ABX.TO...
  → 454 registros guardados en MongoDB.
Procesando BVN...
  → 452 registros guardados en MongoDB.
Procesando BHP...
  → 452 registros guardados en MongoDB.

Resumen final:
  FSM: ✅ OK
  VOLCABC1.LM: ✅ OK
  ABX.TO: ✅ OK
  BVN: ✅ OK
  BHP: ✅ OK


CELDA 8 — Verificación final (lee de vuelta lo que se guardó)

In [ ]:
print("Verificación de la colección 'precios_ohlcv':\n")

for doc in coleccion_precios.find({}, {"ticker": 1, "fechas": 1, "close": 1, "_id": 0}):
    print(f"Ticker: {doc['ticker']}")
    print(f"  Rango de fechas: {doc['fechas'][0]} → {doc['fechas'][-1]}")
    print(f"  Total registros: {len(doc['fechas'])}")
    print(f"  Último precio Close: {doc['close'][-1]}")
    print()

Verificación de la colección 'precios_ohlcv':

Ticker: FSM
  Rango de fechas: 2024-09-12 → 2026-07-02
  Total registros: 452
  Último precio Close: 8.72

Ticker: VOLCABC1.LM
  Rango de fechas: 2024-09-12 → 2026-07-02
  Total registros: 444
  Último precio Close: 0.861

Ticker: ABX.TO
  Rango de fechas: 2024-09-12 → 2026-07-03
  Total registros: 454
  Último precio Close: 55.56

Ticker: BVN
  Rango de fechas: 2024-09-12 → 2026-07-02
  Total registros: 452
  Último precio Close: 29.72

Ticker: BHP
  Rango de fechas: 2024-09-12 → 2026-07-02
  Total registros: 452
  Último precio Close: 83.33

